In [1]:
import polars as pl
import folium

In [2]:
df = pl.read_csv("../data/output/full_filtered_df.csv")


In [3]:
########################
#Missing start or end station number
########################
missing_values = df.null_count()

print(missing_values)

#184 missing values for columns end station number and total duration. if the missing values for the entries are not the same we have a total of 
# 184*2 = 368 out of 9068241 entries. Removing them would be a loss of less than 1% of the dataframe

df_no_nulls = df.drop_nulls()
print(df_no_nulls.shape)
print("---------------")


shape: (1, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ u32        ┆ u32                  ┆ u32      ┆ u32                ┆ u32                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
│ 0          ┆ 0                    ┆ 0        ┆ 184                ┆ 184                 │
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
(9068057, 5)
---------------


In [4]:
########################
#Total duration = 0, missing or negative
########################

df_duration = df_no_nulls.filter(pl.col("Total duration (ms)") <= 0)
print(df_duration)
print("-------------------")

# After removing the missing entries for the total duration, we see there are no negative or 0 ms entries 


shape: (0, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ str        ┆ i64                  ┆ str      ┆ i64                ┆ i64                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘
-------------------


In [5]:
########################
#same start and end station number
########################

#are there entries where the start and end stations are the same?
df_same_station = df_no_nulls.filter(pl.col("Start station number") == pl.col("End station number"))
print(df_same_station)

#305702 entries start and end in the same station. this represents 305702/9068057 = 0.033 -> 3.3% of the no_nulls dataset

#Lets filter them out
df_no_same_station = df_no_nulls.filter(pl.col("Start station number") != pl.col("End station number"))
print(df_no_same_station)

#new df shape is 8_762_355

shape: (305_702, 5)
┌──────────────────┬───────────────┬──────────────────┬──────────────────────┬─────────────────────┐
│ Start date       ┆ Start station ┆ End date         ┆ End station number   ┆ Total duration (ms) │
│ ---              ┆ number        ┆ ---              ┆ ---                  ┆ ---                 │
│ str              ┆ ---           ┆ str              ┆ i64                  ┆ i64                 │
│                  ┆ i64           ┆                  ┆                      ┆                     │
╞══════════════════╪═══════════════╪══════════════════╪══════════════════════╪═════════════════════╡
│ 2025-01-14 23:40 ┆ 300050        ┆ 2025-01-15 00:06 ┆ 300050               ┆ 1559817             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4281047             │
│ 2025-01-14 23:35 ┆ 300059        ┆ 2025-01-15 00:46 ┆ 300059               ┆ 4301835             │
│ 2025-01-14 23:27 ┆ 200166        ┆ 2025-01-14 23:27 ┆ 200166         

In [6]:

########################
#incoherent start and end station numbers (?)
########################

#-------------------------
#Stations appearing in one column but not the other?
#-------------------------

#get uniqueids for both columns
start_ids = df_no_same_station["Start station number"].unique()
end_ids = df_no_same_station["End station number"].unique()

#compare them
start_only = start_ids.filter(~start_ids.is_in(end_ids))

end_only = end_ids.filter(~end_ids.is_in(start_ids))


print("Only in Start:", start_only)
print("Only in End:", end_only)

#There are 2 stations that only appear in the end station number column, station ids 10626 and 22168. should they be removed??

#-------------------------
# Negative station numbers?
#-------------------------

negative_stations = df_no_same_station.filter((pl.col("Start station number") < 0) | (pl.col("End station number") < 0))

print(negative_stations)

# No negative station number ids

Only in Start: shape: (0,)
Series: 'Start station number' [i64]
[
]
Only in End: shape: (2,)
Series: 'End station number' [i64]
[
	10626
	22168
]
shape: (0, 5)
┌────────────┬──────────────────────┬──────────┬────────────────────┬─────────────────────┐
│ Start date ┆ Start station number ┆ End date ┆ End station number ┆ Total duration (ms) │
│ ---        ┆ ---                  ┆ ---      ┆ ---                ┆ ---                 │
│ str        ┆ i64                  ┆ str      ┆ i64                ┆ i64                 │
╞════════════╪══════════════════════╪══════════╪════════════════════╪═════════════════════╡
└────────────┴──────────────────────┴──────────┴────────────────────┴─────────────────────┘


/tmp/ipykernel_22736/3619483098.py:14: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  start_only = start_ids.filter(~start_ids.is_in(end_ids))
/tmp/ipykernel_22736/3619483098.py:16: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  end_only = end_ids.filter(~end_ids.is_in(start_ids))


In [7]:
#######################
# Add date columns
#######################
#Transform start and end date from text to date time
df = df_no_same_station.with_columns(
    pl.col("Start date").str.to_datetime("%Y-%m-%d %H:%M"),
    pl.col("End date").str.to_datetime("%Y-%m-%d %H:%M")
)

#Add Operating date, day of week and hour of day columns
df = df.with_columns(
    pl.col("Start date").dt.date().alias("Operating date"),
    pl.col("Start date").dt.weekday().alias("Day of week"),
    pl.col("Start date").dt.hour().alias("Hour of day"),
)

print(df)

shape: (8_762_355, 8)
┌────────────┬────────────┬────────────┬────────────┬────────────┬────────────┬────────┬───────────┐
│ Start date ┆ Start      ┆ End date   ┆ End        ┆ Total      ┆ Operating  ┆ Day of ┆ Hour of   │
│ ---        ┆ station    ┆ ---        ┆ station    ┆ duration   ┆ date       ┆ week   ┆ day       │
│ datetime[μ ┆ number     ┆ datetime[μ ┆ number     ┆ (ms)       ┆ ---        ┆ ---    ┆ ---       │
│ s]         ┆ ---        ┆ s]         ┆ ---        ┆ ---        ┆ date       ┆ i8     ┆ i8        │
│            ┆ i64        ┆            ┆ i64        ┆ i64        ┆            ┆        ┆           │
╞════════════╪════════════╪════════════╪════════════╪════════════╪════════════╪════════╪═══════════╡
│ 2025-01-14 ┆ 1043       ┆ 2025-01-15 ┆ 200149     ┆ 870468     ┆ 2025-01-14 ┆ 2      ┆ 23        │
│ 23:59:00   ┆            ┆ 00:13:00   ┆            ┆            ┆            ┆        ┆           │
│ 2025-01-14 ┆ 300015     ┆ 2025-01-15 ┆ 300229     ┆ 307181     ┆ 20

In [10]:
#######################
# Add station coordinates
#######################
cord = pl.read_csv("../data/tfl_bike_stations.csv")

print(cord)

#Join start station
df = df.join(cord, left_on='Start station number', right_on = 'station_number', how='left', suffix = 'depart_')
df = df.rename({"latitude": "start_station_latitude",
               "longitude": "start_station_longitude",
               "station_name": "start_station_name"})

#Join end station
df = df.join(cord, left_on='End station number', right_on = 'station_number', how='left')
df = df.rename({"latitude": "end_station_latitude",
               "longitude": "end_station_longitude",
               "station_name": "end_station_name"})

df

shape: (799, 4)
┌────────────────┬───────────┬───────────┬─────────────────────────────────┐
│ station_number ┆ latitude  ┆ longitude ┆ station_name                    │
│ ---            ┆ ---       ┆ ---       ┆ ---                             │
│ i64            ┆ f64       ┆ f64       ┆ str                             │
╞════════════════╪═══════════╪═══════════╪═════════════════════════════════╡
│ 1023           ┆ 51.529163 ┆ -0.10997  ┆ River Street , Clerkenwell      │
│ 1018           ┆ 51.499606 ┆ -0.197574 ┆ Phillimore Gardens, Kensington  │
│ 1012           ┆ 51.521283 ┆ -0.084605 ┆ Christopher Street, Liverpool … │
│ 1013           ┆ 51.530059 ┆ -0.120973 ┆ St. Chad's Street, King's Cros… │
│ 3420           ┆ 51.49313  ┆ -0.156876 ┆ Sedding Street, Sloane Square   │
│ …              ┆ …         ┆ …         ┆ …                               │
│ 200145         ┆ 51.537349 ┆ -0.147154 ┆ Gloucester Avenue, Camden Town  │
│ 1188           ┆ 51.536922 ┆ -0.150181 ┆ London Zoo Car Pa

Start date,Start station number,End date,End station number,Total duration (ms),Operating date,Day of week,Hour of day,start_station_latitude,start_station_longitude,start_station_name,end_station_latitude,end_station_longitude,end_station_name
datetime[μs],i64,datetime[μs],i64,i64,date,i8,i8,f64,f64,str,f64,f64,str
2025-01-14 23:59:00,1043,2025-01-15 00:13:00,200149,870468,2025-01-14,2,23,51.517821,-0.096496,"""Museum of London, Barbican""",51.511542,-0.056667,"""Watney Street, Shadwell"""
2025-01-14 23:59:00,300015,2025-01-15 00:04:00,300229,307181,2025-01-14,2,23,51.472509,-0.122831,"""Binfield Road, Stockwell""",51.469202,-0.119022,"""Sidney Road, Stockwell"""
2025-01-14 23:59:00,1068,2025-01-15 00:10:00,1051,640755,2025-01-14,2,23,51.521113,-0.078869,"""Norton Folgate, Liverpool Stre…",51.534042,-0.086379,"""Shoreditch Park, Hoxton"""
2025-01-14 23:59:00,1159,2025-01-15 00:10:00,1007,642548,2025-01-14,2,23,51.522853,-0.099994,"""Berry Street, Clerkenwell""",51.51477,-0.122219,"""Drury Lane, Covent Garden"""
2025-01-14 23:58:00,200048,2025-01-15 00:14:00,1058,944185,2025-01-14,2,23,51.493978,-0.127554,"""Page Street, Westminster""",51.510212,0.004979,"""Orchard Place, Blackwall Tunne…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…
2025-12-16 00:02:00,200141,2025-12-16 00:16:00,200237,837578,2025-12-16,2,0,51.509158,-0.224103,"""Westfield Ariel Way, White Cit…",51.472817,-0.199783,"""Parson's Green , Parson's Gree…"
2025-12-16 00:01:00,200011,2025-12-16 00:16:00,200250,908383,2025-12-16,2,0,51.519265,-0.021345,"""Furze Green, Bow""",51.520893,-0.051394,"""Cleveland Way, Stepney"""
2025-12-16 00:00:00,300096,2025-12-16 00:07:00,962,432164,2025-12-16,2,0,51.511891,-0.107349,"""Tallis Street, Temple""",51.505569,-0.111606,"""Stamford Street, South Bank"""


In [11]:
#######################
#bike map
#######################

# Chargement des stations
GPS_Point = pl.read_csv("../data/tfl_bike_stations.csv")

GPS_Point = GPS_Point[[
    "station_number",
    "station_name",
    "latitude",
    "longitude"
]]

# Centre approximatif de Londres
london_map = folium.Map(
    location=[51.5074, -0.1278],
    zoom_start=12
)

# Ajout des stations
for row in GPS_Point.drop_nulls(["latitude", "longitude"]).iter_rows(named=True):
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=3,
        popup=f'{row["station_name"]} - {row["station_number"]}',
        fill=True,
        fill_opacity=0.7
    ).add_to(london_map)



# Sauvegarde
london_map.save("stations_londres.html")

london_map


In [12]:
#######################
#export df
#######################

df.write_csv("../data/output/v2/2.clean_df_v2.csv")